# Introduction au RAG (Retrieval Augmented Generation) 

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/C-rBNv5ZbCn1Qe9a-c_RwQ.png" style="width:50%;margin:auto;display:flex" alt="indexing"/>

## Contexte

### Qu’est-ce que le RAG ?
L’une des applications les plus puissantes rendues possibles par les grands modèles de langage (LLM) est la création de chatbots sophistiqués de questions-réponses (Q&R). Il s’agit d’applications capables de répondre à des questions à partir d’informations sources spécifiques. Ces applications utilisent une technique appelée **génération augmentée par la recherche** (*retrieval-augmented generation*, RAG).  
Le RAG est une méthode permettant d’enrichir les connaissances d’un LLM avec des données supplémentaires, qui peuvent être vos propres données.

Les LLM peuvent raisonner sur un large éventail de sujets, mais leurs connaissances sont limitées aux données publiques disponibles jusqu’à la date de fin d’entraînement du modèle. Si vous souhaitez créer des applications d’IA capables de raisonner sur des données privées ou sur des données introduites après la date de coupure du modèle, vous devez enrichir les connaissances du modèle avec les informations spécifiques dont il a besoin. Le processus consistant à apporter et à insérer les informations appropriées dans le prompt du modèle est appelé **RAG**.

LangChain propose plusieurs composants conçus pour faciliter la création d’applications de questions-réponses et, plus généralement, d’applications basées sur le RAG.

### Architecture du RAG
Une application RAG typique se compose de deux éléments principaux :

* **Indexation** : Un pipeline permettant d’ingérer et d’indexer des données provenant d’une source. Cette étape se déroule généralement hors ligne.

* **Recherche et génération** : La chaîne RAG proprement dite prend la requête de l’utilisateur au moment de l’exécution, récupère les données pertinentes à partir de l’index, puis les transmet au modèle.

La séquence complète la plus courante, allant des données brutes à la réponse, ressemble aux exemples suivants.


- **Indexation**
1. **Chargement** : Tout d’abord, vous devez charger vos données. Cela se fait à l’aide des [DocumentLoaders](https://python.langchain.com/docs/how_to/#document-loaders).

2. **Découpage** : Les [séparateurs de texte (Text splitters)](https://python.langchain.com/docs/how_to/#text-splitters) divisent les `Documents` volumineux en segments plus petits. Cela est utile à la fois pour l’indexation des données et pour leur transmission au modèle, car les gros segments sont plus difficiles à rechercher et ne tiennent pas dans la fenêtre de contexte limitée d’un modèle.

3. **Stockage** : Vous avez besoin d’un endroit pour stocker et indexer ces segments afin qu’ils puissent être recherchés ultérieurement. Cela se fait généralement à l’aide d’un [VectorStore](https://python.langchain.com/docs/how_to/#vector-stores) et d’un modèle d’[Embeddings](https://python.langchain.com/docs/how_to/embed_text/).

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/WEE3pjeJvSZP0R7UL7CYTA.png" width="50%" alt="indexation"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/tutorials/rag/)</span>

- **Recherche et génération**
1. **Recherche** : À partir de l’entrée de l’utilisateur, les segments pertinents sont récupérés depuis le stockage à l’aide d’un composant de recherche (*retriever*).
2. **Génération** : Un ChatModel / LLM produit une réponse en utilisant un prompt qui inclut la question et les données récupérées.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/SwPO26VeaC8VTZwtmWh5TQ.png" width="50%" alt="recherche"/> <br>
<span style="font-size: 10px;">[source](https://python.langchain.com/docs/use_cases/question_answering/)</span>


In [2]:
%pip install -q langchain-community langchain-text-splitters langchain-chroma langchain-ollama pypdf chromadb

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.0 requires langchain-core<2.0.0,>=1.0.0, but you have langchain-core 0.3.63 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-openai 1.1.0 requires langchain-core<2.0.0,>=1.1.0, but you have langchain-core 0.3.63 which is incompatible.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.63 which is incompatible.


In [3]:
%pip install -q langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.4.8 which is incompatible.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.9.4 which is incompatible.
langchain-classic 1.0.0 requires langchain-text-splitters<2.0.0,>=1.0.0, but you have langchain-text-splitters 0.3.8 which is incompatible.
langchain-community 0.3.14 requires langchain-core<0.4.0,>=0.3.29, but you have langchain-core 1.4.8 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.9.4 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.4.8 which is incompatible.


In [3]:
%pip install -q langchain-classic

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.4.8 which is incompatible.
langchain 0.3.25 requires langchain-text-splitters<1.0.0,>=0.3.8, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.9.4 which is incompatible.
langchain-community 0.3.14 requires langchain-core<0.4.0,>=0.3.29, but you have langchain-core 1.4.8 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.9.4 which is incompatible.


In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TokenTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_classic.chains import ConversationalRetrievalChain, RetrievalQA
from langchain_classic.memory import ConversationBufferMemory
from langchain_core.documents import Document as LangchainDocument
from typing import List

# Retrieval avancé (composants legacy -> langchain_classic)
from langchain_classic.retrievers import (
    ContextualCompressionRetriever,
    EnsembleRetriever,
    ParentDocumentRetriever,
)
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.merger_retriever import MergerRetriever
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryStore

## 1. Choix du LLM — 100 % local avec Ollama

Ce notebook tourne **entièrement en local**, sur le CPU, **sans clé API ni connexion** (une fois le modèle téléchargé).

**Prérequis (une seule fois) :**
```bash
# 1) Installer Ollama : https://ollama.com  (Linux : curl -fsSL https://ollama.com/install.sh | sh)
# 2) Démarrer le serveur (laisser la fenêtre ouverte) :
ollama serve
# 3) Télécharger un modèle léger :
ollama pull llama3.2:3b     # ~2 Go, bon compromis (défaut)
# Machines très modestes (~4 Go RAM) :
ollama pull llama3.2:1b     # ~1.3 Go, le plus léger
```

**💡 Connexion limitée :** téléchargez le modèle UNE fois sur une machine connectée, puis copiez le dossier `~/.ollama/models` (Windows : `C:\Users\<nom>\.ollama\models`) sur les postes des élèves par clé USB. Idem pour le cache des embeddings : `~/.cache/huggingface`.

Les options HuggingFace Hub / Groq restent dans la cellule de config **à titre indicatif**, mais elles nécessitent internet + une clé : ne les utilisez pas pour l'usage hors-ligne visé ici.

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
# ============================================================
#  Configuration du LLM  (défaut : Ollama, 100% local)
# ============================================================

# >>> Choisissez le modèle ici (un seul mot à changer) <<<
MODEL = "llama3.2:3b"      # machines modestes : "llama3.2:1b"

# --- Option 1 : Ollama (LOCAL, gratuit, hors-ligne) ---
def setup_ollama_llm(model_name=MODEL):
    """Configure un LLM local via Ollama."""
    return ChatOllama(
        model=model_name,
        temperature=0.3,
        num_predict=512,   # longueur max de la réponse
        top_k=10,
        top_p=0.95,
    )

# --- Option 2 : HuggingFace Hub (EN LIGNE, clé requise) — non recommandé ici ---
def setup_huggingface_llm():
    """LLM hébergé chez HuggingFace. Nécessite internet + un token. À éviter en classe."""
    from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
    endpoint = HuggingFaceEndpoint(
        repo_id="mistralai/Mistral-7B-Instruct-v0.2",
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
        temperature=0.3, max_new_tokens=512,
    )
    return ChatHuggingFace(llm=endpoint)

# --- Option 3 : Groq (EN LIGNE, clé requise, très rapide) ---
def setup_groq_llm():
    from langchain_groq import ChatGroq
    return ChatGroq(model="llama3-70b-8192", temperature=0.3,
                    max_tokens=512, api_key=os.getenv("GROQ_API_KEY"))

# >>> Option active <<<
LLM_OPTION = "huggingface"   # "ollama" (local) | "huggingface" | "groq"

if LLM_OPTION == "ollama":
    llm = setup_ollama_llm(MODEL)
    print(f"✅ LLM local configuré avec Ollama ({MODEL})")
elif LLM_OPTION == "huggingface":
    llm = setup_huggingface_llm()
    print("✅ LLM configuré avec HuggingFace (en ligne)")
elif LLM_OPTION == "groq":
    llm = setup_groq_llm()
    print("✅ LLM configuré avec Groq (en ligne)")
else:
    raise ValueError("Option LLM non reconnue")

# Test rapide de connexion au serveur Ollama
if LLM_OPTION == "ollama":
    try:
        print("Test :", llm.invoke("Réponds juste par : OK").content)
    except Exception as e:
        print("⚠️ Ollama ne répond pas. Lancez 'ollama serve' dans un terminal,")
        print("   et vérifiez que le modèle est présent ('ollama list').")
        print("Détail :", e)

✅ LLM configuré avec HuggingFace (en ligne)


## 2. Lecture et préparation des documents

In [5]:
# Chargement du document
filename = "prenoms_beninois.pdf"
documents = PyPDFLoader(filename).load()
print(f"📄 Document chargé avec {len(documents)} pages")

# Afficher un aperçu
print("\nAperçu du document:")
print(documents[0].page_content[:500] + "...")

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 160 0 (offset 0)
Ignoring wrong pointing object 161 0 (offset 0)


📄 Document chargé avec 32 pages

Aperçu du document:
🎙 « Bonjour et bienvenue dans Les mots panés. Ici, chaque mot, chaque expression, est une 
bouchée à savourer, pour en découvrir les sens cachés » 
Je m’appelle Lindagbé, je suis une béninoise, passionnée de mots, de partage. 
Ensemble nous allons explorer un nom, un prénom, une expression, un mot d'une des sublimes 
langues parlées au Bénin.  Mon objectif ? Qu'on se découvre, qu'on se comprenne,  qu’on 
s'émerveille  et surtout qu'on crée des liens à travers ces trésors linguistiques.  C'est pa...


## 3. Techniques avancées de découpage (Chunking)

In [6]:
# 1. Découpage récursif standard
text_splitter_standard = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
chunks_standard = text_splitter_standard.split_documents(documents)

# 3. Découpage avec recouvrement intelligent
text_splitter_overlap = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    add_start_index=True,
    separators=["\n\n", "\n", ".", " ", ""],
)
chunks_overlap = text_splitter_overlap.split_documents(documents)

print(f"📊 Nombre de chunks:")
print(f"  - Standard: {len(chunks_standard)}")
# print(f"  - Sémantique: {len(chunks_semantic)}")
print(f"  - Grand recouvrement: {len(chunks_overlap)}")

📊 Nombre de chunks:
  - Standard: 62
  - Grand recouvrement: 47


## 4. Techniques avancées de Retrieval

In [10]:
!pip install huggingface_hub[hf_xet]

  Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
Using cached hf_xet-1.5.1-cp37-abi3-win_amd64.whl (4.0 MB)


In [7]:
# Configuration des embeddings (open source, locaux, CPU)
# Multilingue : adapté à des données en français (prénoms béninois).
# Téléchargé UNE fois (~470 Mo) puis mis en cache (~/.cache/huggingface).
# Alternative ultra-légère (anglais surtout) : "sentence-transformers/all-MiniLM-L6-v2" (~90 Mo)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Création du VectorStore (en mémoire, aucune base à installer)
vectorstore = Chroma.from_documents(chunks_standard, embeddings)
print("✅ VectorStore créé")

c:\Users\mbial\miniconda3\envs\nkobo\Lib\site-packages\transformers\tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


✅ VectorStore créé


In [8]:
def create_retrievers(vectorstore,chunks_standard, llm):
    """Crée différents types de retrievers pour comparaison"""
    
    retrievers = {}
    
    # 1. Retriever standard (similarité vectorielle)
    retrievers["standard"] = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 4}
    )
    
    # 2. Retriever avec MMR (Maximum Marginal Relevance)
    retrievers["mmr"] = vectorstore.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}
    )
    
    # 3. Multi-Query Retriever (génère plusieurs versions de la question)
    try:
        retrievers["multi_query"] = MultiQueryRetriever.from_llm(
            retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
            llm=llm,
            include_original=True
        )
    except:
        print("⚠️ MultiQueryRetriever non disponible, ignoré")
    
    # 4. Parent Document Retriever (récupère des chunks petits mais retourne les parents)
    try:
        child_splitter = RecursiveCharacterTextSplitter(chunk_size=200)
        store = InMemoryStore()
        retrievers["parent_doc"] = ParentDocumentRetriever(
            vectorstore=vectorstore,
            docstore=store,
            child_splitter=child_splitter,
            search_kwargs={"k": 4}
        )
        # Ajouter les documents au retriever
        retrievers["parent_doc"].add_documents(chunks_standard)
    except:
        print("⚠️ ParentDocumentRetriever non disponible, ignoré")
    
    # 5. Ensemble Retriever (combine plusieurs méthodes)
    try:
        # BM25 (recherche par mots-clés)
        bm25_retriever = BM25Retriever.from_documents(chunks_standard)
        bm25_retriever.k = 4
        
        retrievers["ensemble"] = EnsembleRetriever(
            retrievers=[bm25_retriever, vectorstore.as_retriever(search_kwargs={"k": 4})],
            weights=[0.5, 0.5]
        )
    except:
        print("⚠️ EnsembleRetriever non disponible, ignoré")
    
    return retrievers

In [9]:

retrievers = create_retrievers(vectorstore, chunks_standard, llm)
print(f"✅ {len(retrievers)} retrievers créés:")
for name in retrievers.keys():
    print(f"  - {name}")

⚠️ EnsembleRetriever non disponible, ignoré
✅ 4 retrievers créés:
  - standard
  - mmr
  - multi_query
  - parent_doc


## 5. Test des différentes techniques de retrieval

In [10]:
# Fonction pour tester un retriever
def test_retriever(retriever, query, retriever_name):
    """Test un retriever et retourne les résultats"""
    try:
        # Récupération des documents
        docs = retriever.invoke(query)
        
        print(f"\n🔍 {retriever_name.upper()}:")
        print(f"  - {len(docs)} documents récupérés")
        
        # Afficher les premiers documents
        for i, doc in enumerate(docs[:2]):
            content_preview = doc.page_content[:200].replace('\n', ' ')
            print(f"  - Document {i+1}: {content_preview}...")
        
        return docs
    except Exception as e:
        print(f"❌ Erreur avec {retriever_name}: {e}")
        return []

In [11]:
# Test avec une requête
test_query = "Que signifie le prénom Mindéssè ?"
print(f"\n🧪 Test de retrieval avec la question: '{test_query}'\n")
print("=" * 50)

results = {}
for name, retriever in retrievers.items():
    results[name] = test_retriever(retriever, test_query, name)


🧪 Test de retrieval avec la question: 'Que signifie le prénom Mindéssè ?'


🔍 STANDARD:
  - 4 documents récupérés
  - Document 1: Le prénom à l’honneur : Mindéssè  Mindéssè est un prénom à la fois masculin et féminin que l’on retrouve dans plusieurs langues  parlées au Sud du Bénin, comme le goun.  Que signifie Mindéssè ?...
  - Document 2: proches !  Le prénom à l’honneur : Mindéssè  Mindéssè est un prénom à la fois masculin et féminin que l’on retrouve dans plusieurs langues  parlées au Sud du Bénin, comme le goun....

🔍 MMR:
  - 4 documents récupérés
  - Document 1: Le prénom à l’honneur : Mindéssè  Mindéssè est un prénom à la fois masculin et féminin que l’on retrouve dans plusieurs langues  parlées au Sud du Bénin, comme le goun.  Que signifie Mindéssè ?...
  - Document 2: Djogbénou vient de l’expression : « Djogbénou wè kou » .   ● Djo : naître, venir au monde  ● Gbé : le jour, la journée  ● Nou : chose  ● wè veut dire être, et   ● kou veut dire la mort....

🔍 MULTI_QUERY:
  - 7

## 6. Création de la chaîne RAG avec différentes techniques

In [12]:
# Prompt template avancé
prompt_template = """Tu es un assistant linguistique, tu aide les parents à donner des prénoms béninois à leurs enfants.

Contexte:
{context}

Question: {question}

Règles:
1. Si l'information n'est pas dans le contexte, dis-le clairement
2. Ne pas inventer d'informations
3. Si tu as plusieurs sources, cite-les
4. Sois précis et concis

Réponse:"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Fonction pour créer une chaîne QA avec un retriever donné
def create_qa_chain(retriever, llm, prompt=PROMPT):
    """Crée une chaîne de QA avec un retriever personnalisé"""
    return RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": prompt},
        return_source_documents=False
    )

In [13]:
# Création des chaînes QA
qa_chains = {}
for name, retriever in retrievers.items():
    try:
        qa_chains[name] = create_qa_chain(retriever, llm)
        print(f"✅ Chaîne {name} créée")
    except Exception as e:
        print(f"❌ Erreur avec {name}: {e}")

✅ Chaîne standard créée
✅ Chaîne mmr créée
✅ Chaîne multi_query créée
✅ Chaîne parent_doc créée


In [14]:
# Fonction pour comparer les différentes chaînes
def compare_retrievers(qa_chains, query):
    """Compare les réponses de différentes chaînes QA"""
    print(f"\n📝 Question: {query}\n")
    print("=" * 60)
    
    responses = {}
    for name, chain in qa_chains.items():

        response = chain.invoke(query)
        responses[name] = response['result']
        print(f"\n🔹 {name.upper()}:")
        print(f"  {response['result'][:300]}...")
    
    return responses

In [15]:
# Test de comparaison
test_queries = [
    "Que signifie le prénom Mindéssè ?",
    "Où se trouve Douala?",
    "QQue signifie Sourou ?"
]

for query in test_queries:
    compare_retrievers(qa_chains, query)
    print("\n" + "-" * 60)


📝 Question: Que signifie le prénom Mindéssè ?


🔹 STANDARD:
   Mindéssè is a name of both genders found in several languages spoken in the South of Benin, including the Goun language. The name Mindéssè comes from the phrase "Min dé man gnin min dé sè." Translated, "Min dé" means "a person." Therefore, Mindéssè can be interpreted as "the one who belongs to a pe...

🔹 MMR:
   Mindéssè est un prénom à la fois masculin et féminin que l'on retrouve dans plusieurs langues parlées au Sud du Bénin, comme le goun. Le prénom Mindéssè signifie "ce qui est entre la naissance et la mort" en goun. Les composants du prénom sont :

* Mindé : entre
* Sè : naissance
* Sè : mort

Source...

🔹 MULTI_QUERY:
   Mindéssè is a name of both genders found in several languages spoken in the South of Benin, such as Goun. It comes from the phrase "Min dé man gnin min dé sè." In this phrase, "Min dé" means "a person," "Man" indicates negation, and "Gnin" means "to be." Therefore, Mindéssè can be translated as "a .

## 7. Agent conversationnel avancé

In [16]:
def create_conversational_agent(retriever, llm):
    """Crée un agent conversationnel avec mémoire"""
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )
    
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        chain_type="stuff",
        get_chat_history=lambda h: h,
        return_source_documents=True
    )
    
    return qa

In [17]:
def run_agent(retriever_type="standard", max_turns=5):
    """Exécute l'agent conversationnel"""
    print(f"\n🤖 Agent avec retriever {retriever_type}")
    print("Tape 'quit', 'exit' ou 'bye' pour quitter\n")
    
    # Utiliser le meilleur retriever disponible
    if retriever_type in retrievers:
        retriever = retrievers[retriever_type]
    else:
        retriever = retrievers["standard"]
    
    agent = create_conversational_agent(retriever, llm)
    
    turn_count = 0
    while turn_count < max_turns:
        user_input = input("\n👤 Vous: ")
        
        if user_input.lower() in ["quit", "exit", "bye"]:
            print("🤖 Assistant: Au revoir! Bonne journée!")
            break
        
        try:
            response = agent.invoke({"question": user_input})
            print(f"🤖 Assistant: {response['answer']}")
            turn_count += 1
        except Exception as e:
            print(f"❌ Erreur: {e}")
            break

In [19]:
# Lancer l'agent avec différentes techniques
print("\n🎯 Choix du retriever:")
for i, name in enumerate(retrievers.keys()):
    print(f"  {i+1}. {name}")

# Choisir une technique
selected = "mmr"  # Changez ici pour tester différentes techniques
# selected = "mmr"
# selected = "ensemble"
# selected = "multi_query"

run_agent(selected)


🎯 Choix du retriever:
  1. standard
  2. mmr
  3. multi_query
  4. parent_doc

🤖 Agent avec retriever mmr
Tape 'quit', 'exit' ou 'bye' pour quitter

❌ Erreur: Got multiple output keys: dict_keys(['answer', 'source_documents']), cannot determine which to store in memory. Please set the 'output_key' explicitly.


## 8. Évaluation des performances des différentes techniques

In [20]:
import time
import pandas as pd

def evaluate_retrieval_performance(retriever, queries, k=4):
    """Évalue la performance d'un retriever"""
    results = []
    
    for query in queries:
        start_time = time.time()
        try:
            docs = retriever.invoke(query)
            elapsed = time.time() - start_time
            
            results.append({
                'query': query[:50],
                'num_docs': len(docs),
                'time': elapsed,
                'avg_content_length': sum(len(d.page_content) for d in docs) / len(docs) if docs else 0
            })
        except Exception as e:
            results.append({
                'query': query[:50],
                'num_docs': 0,
                'time': 0,
                'error': str(e)
            })
    
    return pd.DataFrame(results)

In [21]:
# Évaluation des retrievers
eval_queries = [
    "Que signifie le prénom Mindéssè ?",
    "Quels prénoms béninois signifient la joie ?",
    "Donne un prénom lié à Dieu",
    "Que signifie Sourou ?",
    "Quels prénoms sont liés à la naissance ?",
]

performance_data = {}
for name, retriever in retrievers.items():
    print(f"\nÉvaluation de {name}...")
    try:
        df = evaluate_retrieval_performance(retriever, eval_queries)
        performance_data[name] = df
        print(f"  ✅ {len(df)} requêtes évaluées")
    except Exception as e:
        print(f"  ❌ Erreur: {e}")


Évaluation de standard...
  ✅ 5 requêtes évaluées

Évaluation de mmr...
  ✅ 5 requêtes évaluées

Évaluation de multi_query...
  ✅ 5 requêtes évaluées

Évaluation de parent_doc...
  ✅ 5 requêtes évaluées


In [22]:
# Résumé des performances
summary = []
for name, df in performance_data.items():
    if not df.empty and 'time' in df.columns:
        summary.append({
            'Technique': name,
            'Temps moyen (s)': df['time'].mean(),
            'Documents récupérés': df['num_docs'].mean(),
            'Temps total (s)': df['time'].sum()
        })

if summary:
    summary_df = pd.DataFrame(summary)
    print("\n📊 Résumé des performances:")
    print(summary_df.to_string(index=False))


📊 Résumé des performances:
  Technique  Temps moyen (s)  Documents récupérés  Temps total (s)
   standard         0.032401                  4.0         0.162006
        mmr         0.038198                  4.0         0.190990
multi_query         5.718564                  8.2        28.592822
 parent_doc         0.017001                  3.2         0.085004


## 9. Configuration et installation

In [ ]:
print("\n🔧 Dépendances Python (une fois) :")
print("""
pip install langchain langchain-community langchain-classic
pip install langchain-text-splitters langchain-chroma langchain-huggingface langchain-ollama
pip install chromadb pandas sentence-transformers pypdf python-dotenv
""")

print("\n📦 Ollama (LLM local, hors-ligne) :")
print("""
Installation : https://ollama.com
Démarrer     : ollama serve   (laisser ouvert)
Modèles      :
  - ollama pull llama3.2:3b   (~2 Go, défaut)
  - ollama pull llama3.2:1b   (~1.3 Go, machines modestes)
Hors-ligne   : copier ~/.ollama/models et ~/.cache/huggingface par clé USB
""")

In [ ]:
# Sauvegarde pour utilisation future
import pickle

def save_retriever(retriever, filename):
    """Sauvegarde un retriever"""
    with open(filename, 'wb') as f:
        pickle.dump(retriever, f)
    print(f"✅ Retriever sauvegardé dans {filename}")

def load_retriever(filename):
    """Charge un retriever"""
    with open(filename, 'rb') as f:
        return pickle.load(f)

# Sauvegarder le meilleur retriever
best_retriever_name = "ensemble"  # ou "standard", "mmr", etc.
if best_retriever_name in retrievers:
    save_retriever(retrievers[best_retriever_name], "best_retriever.pkl")
    print(f"\n💾 Le retriever '{best_retriever_name}' a été sauvegardé")